# DRHP Drafting Agent - Capital Structure Section

This notebook demonstrates how to build a production-ready DRHP (Draft Red Herring Prospectus) drafting system using **Syrin**.

## Features Demonstrated

- **Knowledge**: Document ingestion from multiple sources (PAS-3, SH-7, MOA)
- **Grounding**: Fact extraction and verification
- **Structured Output**: Pydantic models for DRHP data
- **Budget Management**: Cost limits per run
- **Guardrails**: FactVerificationGuardrail for output validation
- **Tools Integration**: search_knowledge, search_knowledge_deep

## Prerequisites

```bash
pip install syrin[openai]
```

Set your OpenAI API key:
```python
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"
```

In [17]:
import syrin

syrin.__version__

'0.8.0'

## Step 1: Explore Input Data

Let's first look at the input documents provided in the `data/` directory.

In [3]:
from pathlib import Path

# Define data directory
DATA_DIR = Path(
    "/Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data"
)

# List all files
print("=== Input Documents ===\n")
for f in sorted(DATA_DIR.rglob("*.md")):
    print(f"  📄 {f.relative_to(DATA_DIR)}")

=== Input Documents ===

  📄 PAS-3/Board Resolution Allotment of Shares.md
  📄 PAS-3/List of Allottees.md
  📄 PAS-3/PAS-3 Form.md
  📄 SH-7/Board Meeting.md
  📄 SH-7/EGM.md
  📄 SH-7/MOA.md
  📄 SH-7/Notice of EGM.md
  📄 SH-7/SH7.md


### Sample Document Content

Let's look at a sample document to understand the data format.

In [4]:
# Read sample document
sample_file = DATA_DIR / "PAS-3" / "List of Allottees.md"
print(f"=== {sample_file.name} ===\n")
print(sample_file.read_text()[:1500] + "...")

=== List of Allottees.md ===

NEXUS BRIGHTLEARN SOLUTIONS PVT LTD
Krishna Tower, GF, Sector-12 Dwarka, New Delhi 110075
List of Allottees
Table A
| Name of the Company                                                          | Nexus Brightlearn Solutions Private Limited   |
|-|-|
| Date of the allotment                                                        | 15 May, 2019                          |
| Type of share allotted (Equity/Preference)                                   | Equity Shares                              |
| Nominal value per share                                                      | Rs.10/-each                                |
| Premium amount per share (in Rs.).                                           | Rs. 7,850                                  |
| Total number of allottees                                                    | Three                                      |
| Brief particulars in respect of terms and condition, voting<br />rights, etc | Shall rank pa

## Step 2: Create Knowledge Base

Now let's create the Knowledge base with all the documents. This uses Syrin's:
- **Embedding**: text-embedding-3-small for semantic search
- **Chunking**: RECURSIVE strategy with 512 char chunks
- **Agentic RAG**: Multi-query search with decomposition
- **Grounding**: Fact extraction and verification

In [5]:
import sys

sys.path.insert(0, "/Users/devscript/Documents/Syrin-Labs/syrin-python")
import nest_asyncio

nest_asyncio.apply()
import os
from pathlib import Path

from dotenv import load_dotenv

os.environ.setdefault("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY", ""))

# Load .env file
load_dotenv(
    "/Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/.env",
    override=True,
)

from syrin.embedding import Embedding
from syrin.enums import KnowledgeBackend
from syrin.knowledge import (
    AgenticRAGConfig,
    ChunkConfig,
    ChunkStrategy,
    GroundingConfig,
    Knowledge,
)

# Set API key
os.environ.setdefault("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY", ""))

# Define paths
DATA_DIR = Path(
    "/Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data"
)


def create_knowledge() -> Knowledge:
    """Create Knowledge orchestrator with document sources."""
    embedding = Embedding.OpenAI("text-embedding-3-small")

    sources = [Knowledge.Directory(str(DATA_DIR), glob="**/*.md")]

    return Knowledge(
        sources=sources,
        embedding=embedding,
        backend=KnowledgeBackend.MEMORY,
        chunk_config=ChunkConfig(
            strategy=ChunkStrategy.RECURSIVE,
            chunk_size=512,
            min_chunk_size=64,
        ),
        top_k=12,
        score_threshold=0.2,
        agentic=True,
        agentic_config=AgenticRAGConfig(
            max_search_iterations=3,
            decompose_complex=True,
            grade_results=True,
            relevance_threshold=0.35,
            web_fallback=False,
        ),
        grounding=GroundingConfig(
            extract_facts=True,
            verify_before_use=True,
            cite_sources=True,
            confidence_threshold=0.7,
        ),
        inject_system_prompt=True,
    )


# Create knowledge base
print("Creating Knowledge base...")
knowledge = create_knowledge()
print(f"✅ Knowledge created with {len(knowledge._sources)} sources")

Creating Knowledge base...
✅ Knowledge created with 1 sources


## Step 3: Explore Knowledge

Let's explore what the Knowledge base contains.

In [6]:
# Check knowledge state
print("=== Knowledge State ===")
print(f"Sources: {len(knowledge._sources)}")
print(f"Embedding: {knowledge._embedding.model_id}")
print("Knowledge base ready for queries")

=== Knowledge State ===
Sources: 1
Embedding: text-embedding-3-small
Knowledge base ready for queries


## Step 4: Create the DRHP Agent

Now let's create the Agent with all Syrin features:
- **Model**: gpt-4o for generation
- **System Prompt**: Legal drafting instructions
- **Knowledge**: RAG with grounding
- **Structured Output**: DraftOutput schema
- **Budget**: $1.00 per run
- **Guardrails**: FactVerificationGuardrail

In [7]:
from examples.ipo_drafting_agent.ipo_agent.models import DraftOutput

from syrin import Agent, Budget, FactVerificationGuardrail, Output
from syrin.model import Model

SYSTEM_PROMPT = """You are a DRHP drafting assistant for Capital Structure and Shareholding Pattern.

CRITICAL - Call search_knowledge FIRST. Use these queries: "capital structure", "authorized capital", "issued capital", "shareholding", "PAS-3", "List of Allottees", "allottees", "equity shares", "MOA", "SH-7". Do NOT answer until you have retrieved facts.

RULES:
- Copy numbers and names EXACTLY from search results. Do not invent or approximate.
- Use the company name exactly as it appears in the documents.
- For shareholding: Use the latest shareholding pattern from the documents.
- draft_section MUST be paragraph-style legal disclosure using ONLY facts from search results.
- Do NOT use example company names or invented numbers. Use facts from retrieved documents only.

Output a single valid JSON object (no markdown, no code blocks):
{
  "draft_section": "<paragraph in formal legal language>",
  "sources_used": ["<doc names used>"],
  "auto_extracted_parts": ["<fields extracted from docs>"],
  "requires_review": ["<items needing human verification>"]
}"""


def create_agent(knowledge: Knowledge) -> Agent:
    """Create the DRHP drafting agent."""
    return Agent(
        name="ipo_drafting_agent",
        description="A DRHP drafting assistant for Capital Structure and Shareholding Pattern.",
        model=Model.OpenAI("gpt-4o", api_key=os.getenv("OPENAI_API_KEY")),
        system_prompt=SYSTEM_PROMPT,
        knowledge=knowledge,
        output=Output(DraftOutput, validation_retries=3),
        max_tool_iterations=15,
        budget=Budget(run=1.0),
        guardrails=[FactVerificationGuardrail()],
    )


# Create agent
print("Creating DRHP Agent...")
agent = create_agent(knowledge)
print(f"✅ Agent: {agent.name}")
print(f"   Model: {agent._model._model_id}")
print(f"   Budget: ${agent._budget.run if agent._budget else 'None'}")
print(f"   Guardrails: {len(agent._guardrails)}")

Creating DRHP Agent...
✅ Agent: ipo_drafting_agent
   Model: openai/gpt-4o
   Budget: $1.0
   Guardrails: 1


## Step 5: Run the Agent

Now let's run the agent to generate the DRHP Capital Structure section. This will:
1. Search knowledge for relevant documents
2. Extract facts from search results
3. Verify facts against sources
4. Generate structured output

In [8]:
# Run the agent
prompt = "Draft the Capital Structure and Shareholding Pattern section for the DRHP."

print("=" * 70)
print("RUNNING AGENT")
print("=" * 70)
print(f"\nPrompt: {prompt}\n")

result = agent.response(prompt)

print("=" * 70)
print("RESULT RECEIVED")
print("=" * 70)

RUNNING AGENT

Prompt: Draft the Capital Structure and Shareholding Pattern section for the DRHP.



Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/devscript/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1076ebf80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/devscript/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1076ebf80> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-24' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/devscript/Documents/Syrin-Labs/syrin-python/.venv/lib/python3.11/site-package

RESULT RECEIVED


## Step 6: Display Results

Let's display the structured output from the agent.

In [9]:
print("=" * 70)
print("DRAFT SECTION")
print("=" * 70)

if result.structured:
    print(result.structured.parsed.draft_section)
else:
    print(result.content)

DRAFT SECTION
The capital structure of the company has undergone recent changes as reflected in the latest amendments made to the Memorandum of Association and various resolutions. The authorized share capital of the company is now Rs. 300,000 divided into 30,000 equity shares with a nominal value of Rs. 10 per share. This represents an increase from the previous authorized capital which was Rs. 150,000 comprising 15,000 equity shares.

The issued and subscribed capital of the company stands at Rs. 150,000, consisting of 15,000 equity shares, each with a nominal value of Rs. 10. The entire issued and subscribed capital has been fully paid up.

As per the most recent share allotment, three allottees were issued equity shares inclusive of a share premium of INR 7,850 per share. Mr. Rohit Sharma and Mr. Suresh Sharma were each issued 192 equity shares, while Sunrise Engineering Works Pvt. Ltd. received 576 shares. The issued equity shares will rank pari-passu with existing shares, conferr

### Sources Used

In [10]:
print("=" * 70)
print("SOURCES USED")
print("=" * 70)

if result.structured:
    for src in result.structured.parsed.sources_used:
        print(f"  📄 {src}")

SOURCES USED
  📄 /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/SH-7/SH7.md
  📄 /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/SH-7/EGM.md
  📄 /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/PAS-3/Board Resolution Allotment of Shares.md
  📄 /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/PAS-3/List of Allottees.md


### Auto-Extracted Parts

In [11]:
print("=" * 70)
print("AUTO-EXTRACTED PARTS")
print("=" * 70)

if result.structured:
    for part in result.structured.parsed.auto_extracted_parts:
        print(f"  ✓ {part}")

AUTO-EXTRACTED PARTS
  ✓ Authorized capital of the company: Rs. 300,000
  ✓ Number of equity shares in authorized capital: 30,000
  ✓ Nominal value per equity share: Rs. 10
  ✓ Issued and subscribed capital: Rs. 150,000
  ✓ Number of issued equity shares: 15,000
  ✓ Recent share allotment: 3 allottees including Mr. Rohit Sharma, Mr. Suresh Sharma, and Sunrise Engineering Works Pvt. Ltd.


### Items Requiring Review

In [12]:
print("=" * 70)
print("REQUIRES REVIEW")
print("=" * 70)

if result.structured:
    for item in result.structured.parsed.requires_review:
        print(f"  ⚠ {item}")

REQUIRES REVIEW
  ⚠ Verification of any updates post-document retrieval date


## Step 7: Grounding Information

Let's check what facts were grounded and verified.

In [13]:
print("=" * 70)
print("GROUNDED FACTS")
print("=" * 70)

grounded_facts = getattr(agent._runtime, "grounded_facts", None)

if grounded_facts:
    print(f"Total Facts: {len(grounded_facts)}\n")

    verified = sum(1 for f in grounded_facts if f.verification.value == "VERIFIED")
    contradicted = sum(1 for f in grounded_facts if f.verification.value == "CONTRADICTED")
    not_found = sum(1 for f in grounded_facts if f.verification.value == "NOT_FOUND")
    unverified = sum(1 for f in grounded_facts if f.verification.value == "UNVERIFIED")

    print(f"Verified:   {verified}")
    print(f"Contradicted: {contradicted}")
    print(f"Not Found:  {not_found}")
    print(f"Unverified: {unverified}\n")

    print("Sample Facts:")
    for i, fact in enumerate(grounded_facts[:5]):
        print(f"\n  [{i + 1}] {fact.verification.value}")
        print(f"      {fact.content[:100]}...")
        if fact.source:
            print(f"      Source: {fact.source}")
else:
    print("No grounded facts available.")

GROUNDED FACTS
Total Facts: 50

Verified:   50
Contradicted: 0
Not Found:  0
Unverified: 0

Sample Facts:

  [1] VERIFIED
      Revised capital structure after taking into consideration the changes vide points 4, 5, 6 and 8 abov...
      Source: /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/SH-7/SH7.md

  [2] VERIFIED
      value, character and circumstances of any business concerns and undertakings and generally of any as...
      Source: /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/SH-7/MOA.md

  [3] VERIFIED
      The additional capital (taking into consideration the addition above) is divided as follows

(a) Num...
      Source: /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/data/SH-7/SH7.md

  [4] VERIFIED
      TO INCREASE THE AUTHORISED CAPITAL
The Company is having an existing authorized share capital of Rs....
      Source: /Users/devscript/Documents/Syrin-Labs/syrin-py

## Step 8: Budget & Cost Tracking

In [14]:
print("=" * 70)
print("BUDGET & COST")
print("=" * 70)

print(f"\nTotal Cost: ${result.cost:.4f}")

if agent._budget:
    print(f"Budget Limit: ${agent._budget.run}")
    print(f"Budget Spent: ${agent._budget._spent:.4f}")
    print(f"Budget Remaining: ${agent._budget.remaining:.4f}")
    # Budget state tracking available

BUDGET & COST

Total Cost: $0.0688
Budget Limit: $1.0
Budget Spent: $0.2948
Budget Remaining: $0.7052


## Step 8B: Save Intermediate Output

Let's save the intermediate output with grounded facts for inspection.

In [15]:
import json
from pathlib import Path


def shorten_path(p: str) -> str:
    """Shorten path to be more readable."""
    if "examples/ipo_drafting_agent/data/" in p:
        return p.split("examples/ipo_drafting_agent/data/")[-1]
    return p


if grounded_facts:
    intermediate = {
        "prompt": prompt,
        "draft_section": result.structured.parsed.draft_section if result.structured else "",
        "sources_used": [
            shorten_path(s)
            for s in (result.structured.parsed.sources_used if result.structured else [])
        ],
        "auto_extracted_parts": result.structured.parsed.auto_extracted_parts
        if result.structured
        else [],
        "requires_review": result.structured.parsed.requires_review if result.structured else [],
        "grounded_facts": [
            {
                "content": f.content[:500] + "..." if len(f.content) > 500 else f.content,
                "source": shorten_path(f.source) if f.source else "",
                "confidence": round(f.confidence, 3),
                "verification": f.verification.value,
            }
            for f in grounded_facts
        ],
        "metrics": {
            "cost": round(result.cost, 4),
            "total_facts": len(grounded_facts),
            "verified": verified,
            "contradicted": contradicted,
            "not_found": not_found,
            "unverified": unverified,
        },
    }

    output_path = Path(
        "/Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/output_intermediate.json"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(intermediate, indent=2))
    print(f"✅ Intermediate output saved to: {output_path}")
    print(f"   Total facts: {len(grounded_facts)}")
    print(f"   Verified: {verified}/{len(grounded_facts)}")

✅ Intermediate output saved to: /Users/devscript/Documents/Syrin-Labs/syrin-python/examples/ipo_drafting_agent/output_intermediate.json
   Total facts: 50
   Verified: 50/50


## Step 9: Raw Response

Here's the raw response content for reference.

In [16]:
print("=" * 70)
print("RAW RESPONSE (First 2000 chars)")
print("=" * 70)
print(result.content[:2000])

RAW RESPONSE (First 2000 chars)
{"draft_section":"The capital structure of the company has undergone recent changes as reflected in the latest amendments made to the Memorandum of Association and various resolutions. The authorized share capital of the company is now Rs. 300,000 divided into 30,000 equity shares with a nominal value of Rs. 10 per share. This represents an increase from the previous authorized capital which was Rs. 150,000 comprising 15,000 equity shares.\n\nThe issued and subscribed capital of the company stands at Rs. 150,000, consisting of 15,000 equity shares, each with a nominal value of Rs. 10. The entire issued and subscribed capital has been fully paid up.\n\nAs per the most recent share allotment, three allottees were issued equity shares inclusive of a share premium of INR 7,850 per share. Mr. Rohit Sharma and Mr. Suresh Sharma were each issued 192 equity shares, while Sunrise Engineering Works Pvt. Ltd. received 576 shares. The issued equity shares will rank 

## Summary

This example demonstrated how to build a DRHP drafting agent using Syrin with:

### Features Used

| Component | Purpose |
|-----------|---------|
| `Knowledge` | Document ingestion, chunking, embedding, vector store |
| `Agent` | LLM orchestration, tool execution, structured output |
| `search_knowledge` | Semantic RAG retrieval |
| `Output(DraftOutput)` | Structured output validation |
| `GroundingConfig` | Fact extraction and verification |
| `FactVerificationGuardrail` | Output validation |
| `Budget` | Cost management |

### Output Artifacts

- **Draft Section**: Formal legal disclosure language
- **Sources Used**: Document attribution
- **Auto-Extracted Parts**: Fields extracted from documents
- **Requires Review**: Items needing human verification
- **Grounded Facts**: Verified facts with sources
- **Cost Tracking**: Budget spent vs remaining